In [78]:
import numpy as np

In [79]:
#Función de activación
def sigmoide(z):
    return 1/(1 + np.exp(-z))


In [80]:
def sigmoide_deriv(a):
    return a *(1-a)

In [81]:
class Neurona:
    def __init__(self ,seed=0 ,lr=0.1 ):
        self.seed = seed
        self.w = None
        self.b = 0.0
        self.lr = lr
    
    def predecir(self ,X):
        return sigmoide(X @ self.w + self.b)
                      #(4,2)@(2,) + 0.0 = (4,)
                      
    def entrenar(self ,X ,y ,epocas=5000):
        if self.w is None:
            rng = np.random.default_rng(self.seed)
            self.w = rng.normal(0,1,X.shape[1]) #(2,)
        for _ in range(epocas):
            a = self.predecir(X)
            #Derivada de L respecto a z
            dz = a - y
            #Derivada de L respecto a w
            dw = (X.T @ dz)/X.shape[0] 
                 #(2,4)*(4,1)
            db = np.mean(dz)
            
            self.w -= self.lr* dw
            self.b -= self.lr * db


AND

In [82]:
Neurona_1 = Neurona()
X =np.array([[0,0],
            [0,1],
            [1,0],
            [1,1]])
y = np.array([0,0,0,1])
Neurona_1.entrenar(X,y)

In [ ]:
r = Neurona_1.predecir(np.array([
            [0,0],
            [0,1],
            [1,0],
            [1,1]]))
r = (r >= 0.5).astype(int)
r

XOR

In [84]:
Neurona_2 = Neurona()
X =np.array([[0,0],
            [0,1],
            [1,0],
            [1,1]])
y = np.array([0,1,1,0])
Neurona_2.entrenar(X,y)

In [ ]:
r = Neurona_2.predecir(np.array([
            [0,0],
            [0,1],
            [1,0],
            [1,1]]))
r = (r >= 0.5).astype(int)
r

In [86]:
class RedNeuronal:
    def __init__(self ,capas ,seed=0 ,lr=0.1):
        #[2,4,1]
        self.lr = lr
        self.seed = seed
        rng = np.random.default_rng(self.seed)
        self.w = [rng.normal(0,1,(capas[i],capas[i+1])) for i in range(len(capas)-1)]
        self.b = [np.zeros((1, capas[i+1])) for i in range(len(capas)-1)]
    
    def forward(self ,X):
        activaciones = [X]
        for W ,b in zip(self.w , self.b):
            activaciones.append(sigmoide(activaciones[-1] @ W + b))
        return activaciones
        
    def predecir(self ,X):
        return self.forward(X)[-1]
    
    def entrenar(self ,X ,y ,epocas=10000):
        for _ in range(epocas):
            act = self.forward(X)
            dz = act[-1] - y
            
            for i in reversed(range(len(self.w))):
                dw = (act[i].T @ dz)/X.shape[0]
                db = np.mean(dz,axis=0)
                if i > 0:
                    dz = (dz @ self.w[i].T) * sigmoide_deriv(act[i])
                self.w[i] -= self.lr * dw
                self.b[i] -= self.lr * db
            


In [87]:
red_neuronal = RedNeuronal([2,4,1])
X =np.array([[0,0],
            [0,1],
            [1,0],
            [1,1]])
y = np.array([0,1,1,0]).reshape(-1,1)
red_neuronal.entrenar(X,y,epocas=10000)


In [ ]:
r = red_neuronal.predecir(np.array([
            [0,0],
            [0,1],
            [1,0],
            [1,1]]))
r = (r >= 0.5).astype(int)
r     